In [6]:
from google.colab import files
uploaded = files.upload()

Saving bodyfat.csv to bodyfat.csv


In [8]:
import polars as pl
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV
from sklearn.metrics import mean_squared_error, r2_score


df = pl.read_csv("bodyfat.csv")


X_pl = df.drop("bodyfat")
y_pl = df["bodyfat"]

X = X_pl.to_pandas()
y = y_pl.to_numpy()


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# create the 3 models so that they are easy to fit later
models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Lasso CV": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LassoCV(cv=5, random_state=42))
    ]),

    "Ridge CV": Pipeline([
        ("scaler", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-3, 3, 100)))
    ])
}

# fit and evaluate
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "model": name,
        "rmse": rmse,
        "r2": r2
    })

results_pl = pl.DataFrame(results).sort("rmse")

print("Model comparison:")
print(results_pl)

# grab the best model
best_model_name = results_pl.row(0)[0]
best_model = models[best_model_name]

print(f"\nBest model: {best_model_name}")

# refit the best model
best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

best_rmse = np.sqrt(mean_squared_error(y_test, best_pred))
best_r2 = r2_score(y_test, best_pred)

# cute little comparison table
print(f"Best RMSE: {best_rmse:.3f}")
print(f"Best R^2:  {best_r2:.3f}")

Model comparison:
shape: (3, 3)
┌───────────────────┬──────────┬──────────┐
│ model             ┆ rmse     ┆ r2       │
│ ---               ┆ ---      ┆ ---      │
│ str               ┆ f64      ┆ f64      │
╞═══════════════════╪══════════╪══════════╡
│ Lasso CV          ┆ 0.166541 ┆ 0.999404 │
│ Linear Regression ┆ 0.211191 ┆ 0.999041 │
│ Ridge CV          ┆ 0.217056 ┆ 0.998987 │
└───────────────────┴──────────┴──────────┘

Best model: Lasso CV
Best RMSE: 0.167
Best R^2:  0.999
